## Step 1: Raw data gathering- Boston Housing

In [ ]:
!pip install numpy
!pip install pandas
!pip install matplotlib
!pip install scikit-learn
!pip install joblib
!pip install ipywidgets
!pip install openpyxl

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import io

uploaded = files.upload()

filename = next(iter(uploaded))

dataset = pd.read_excel(io.BytesIO(uploaded[filename]), header=1)

dataset.head()

## Step 2: Exploratory Data Analysis (EDA)

In [ ]:
print("Features (Columns):")
print(dataset.columns)

In [ ]:
print(dataset.info())

In [ ]:

# Print all values in the dataset
print(dataset.values)

In [ ]:
# 'PM2.5' is the column to print
column_to_print = None
for col in dataset.columns:
    if 'Pm2.5' in col:
        column_to_print = col
        break

if column_to_print is None:
    raise KeyError("Pm2.5 column not found in dataset.")

# Print the specified column
print(dataset[column_to_print])


In [ ]:
# Correlation matrix for all numerical features
corr_matrix = dataset.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
plt.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Correlation')

plt.xticks(
    range(len(corr_matrix.columns)),
    corr_matrix.columns,
    rotation=90
)

plt.yticks(
    range(len(corr_matrix.columns)),
    corr_matrix.columns
)

plt.title('Correlation Matrix of All Features')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of each feature with PM2.5
p_m_2_5_column_name = None
for col in corr_matrix.columns:
    if 'Pm2.5' in col:
        p_m_2_5_column_name = col
        break

if p_m_2_5_column_name is None:
    raise KeyError("Pm2.5 column not found in corr_matrix.")

important_variables = (
    corr_matrix[p_m_2_5_column_name]
    .drop(p_m_2_5_column_name)
    .sort_values(key=abs, ascending=False)
)

print(important_variables)

In [ ]:
# Visualize variables according to correlation with PM2.5

important_variables.plot(
    kind='bar',
    figsize=(10, 5)
)

plt.title('Correlation of Features with contaminant PM2.5')
plt.xlabel('Features')
plt.ylabel('Correlation')
plt.axhline(0)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))

plt.scatter(dataset['Pm10 （ug/m³）'], dataset['Pm2.5（ug/m³）'])

plt.xlabel('PM10 levels')
plt.ylabel('PM2.5 Levels')
plt.title('Correlation between PM10 Levels and PM2.5 Levels')

In [ ]:
plt.figure(figsize=(6, 4))

plt.scatter(dataset['Rainfall (mm)'], dataset['Pm2.5（ug/m³）'])

plt.xlabel('Rainfall')
plt.ylabel('PM2.5 Levels')
plt.title('Correlation between Rainfall and PM2.5 Levels')

## Step 3: Data Cleansing/ Data pre-processing

In [ ]:
#to check the data type of each field
dataset.info()

In [ ]:
#to check to see if there are any missing values

print(dataset.isnull().sum())

In [ ]:
x = pd.DataFrame(np.c_[dataset['Rainfall (mm)'], dataset['Pm10 （ug/m³）']], columns = ['Rainfall (mm)','Pm10 （ug/m³）'])
Y = dataset['Pm2.5（ug/m³）']

## Step 4: Splitting the Data

In [ ]:
#split the dataset into 70 percent for training and 30 percent for testing

from sklearn.model_selection import train_test_split
x_train, x_test, Y_train, Y_test = train_test_split(x, Y, test_size = 0.3, random_state=5)


In [ ]:
#the testing set has 2969 rows
print(x_test.shape)
print(Y_test.shape)


In [ ]:
#training (70%) = 6927 rows, testing (30%) = 2969 rows, total = 9896
print(x_train.shape)
print(Y_train.shape)


## Step 5: Training the Model on Datasets

In [ ]:
#train model

from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(x_train, Y_train)


In [ ]:
#we will use the testing set to perform some predictions:

pm2_5_pred = model.predict(x_test)
print (pm2_5_pred)


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

# Input boxes
lstat_input = widgets.FloatText(
    value=10.0,
    description='Rainfall (mm):'
)

rm_input = widgets.FloatText(
    value=6.0,
    description='Pm10 (ug/m³):'
)

# Prediction button
predict_button = widgets.Button(
    description='Predict PM2.5 Levels'
)

# Output area
output = widgets.Output()

def predict_price(button):
    with output:
        clear_output()

        # IMPORTANT: column names must be exactly the same
        # as those used when training the model
        new_data = pd.DataFrame(
            [[lstat_input.value, rm_input.value]],
            columns=['Rainfall (mm)', 'Pm10 （ug/m³）']
        )

        prediction = model.predict(new_data)

        print("Predicted PM2.5 Level:", round(prediction[0], 2))

predict_button.on_click(predict_price)

display(lstat_input)
display(rm_input)
display(predict_button)
display(output)

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

# Get uploaded filename
filename = list(uploaded.keys())[0]

# Read uploaded excel, using the second row (index 1) as the header
new_data = pd.read_excel(filename, header=1)

print("Uploaded data:")
display(new_data)

In [ ]:
# Generate predictions
prediction = model.predict(new_data[['Rainfall (mm)', 'Pm10 （ug/m³）']])

# Fill the 'Pm2.5（ug/m³）' column in new_data with predictions
# If the column does not exist, it will be created. If it exists and is empty, it will be filled.
new_data['Pm2.5（ug/m³）'] = prediction

print("Updated New Data with Predictions:")
display(new_data)

In [ ]:
# Save prediction results to Excel sheet

output_filename = 'prediction_results.xlsx'

prediction_results.to_excel(
    output_filename,
    index=False
)

files.download(output_filename)

## Step 6: Model evaluation

In [ ]:
#aim for a value of R-Squared that is close to 1:

print('R-squared: %.4f' % model.score(x_test, Y_test))


In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

mse = mean_squared_error(Y_test, pm2_5_pred)
rmse = np.sqrt(mse)

print("RMSE:", rmse)

## Step 7: Model Export

In [ ]:
import joblib
joblib.dump(model, 'linear_regression_model.pkl')

In [ ]:
# ===================== REUSE AND PREDICT WITH TRAINED MODEL =====================

import numpy as np
import pandas as pd
import joblib

# 1️⃣ Reuse existing trained model (or load from file if needed)
try:
    estimator = model  # change if your variable name is lin_reg, reg, etc.
    print("Using model from current notebook session.")
except NameError:
    estimator = joblib.load("linear_regression_model.joblib")
    print("Loaded model from file: linear_regression_model.joblib")

# 2️⃣ Test prediction using existing test data
try:
    y_pred = estimator.predict(X_test)
    print("\nPrediction on test data (first 5 samples):")
    print("Predicted:", np.round(y_pred[:5], 3))
    print("Actual   :", np.round(y_test[:5], 3))
except Exception as e:
    print("Cannot predict on X_test:", e)

# 3️⃣ Predict a new sample manually
#    👉 Option A: If your model was trained on a DataFrame with named columns
try:
    if hasattr(X_test, "columns"):
        feature_names = list(X_test.columns)
        print("\nFeature names:", feature_names)
        # Example: fill in your own values below
        new_data = pd.DataFrame([{
            feature_names[0]: 3.5,
            feature_names[1]: 2.1,
            # add more features if needed
        }])
        new_pred = estimator.predict(new_data)
        print("\nPrediction for new sample (dict-based):", new_pred)
    else:
        # Option B: For numpy array style training
        new_sample = np.array([[3.5, 2.1]])  # replace with your feature values
        new_pred = estimator.predict(new_sample)
        print("\nPrediction for new sample (list-based):", new_pred)
except Exception as e:
    print("Error making new prediction:", e)

# 4️⃣ (Optional) Save model again if retrained
try:
    joblib.dump(estimator, "linear_regression_model.joblib")
    print("\nModel saved to linear_regression_model.joblib ✅")
except Exception as e:
    print("Model not saved:", e)


## Step 8: Model Deployment

Pre-requisites:

Run the following commands

pip install streamlit pandas scikit-learn

To deploy, make sure the linear_regression_model_pm2_5.pkl and dashboard_fixed,py are in the same directory. After opening a terminal in the directory, run the following command.

streamlit run dashboard_fixed.py

or 

python -m streamlit run dashboard_fixed.py